## 1. Inspeção Inicial:
Carregue a base de dados e realize uma exploração inicial. Informe a quantidade de registros, o nome e tipo de dado de cada coluna, e identifique possíveis problemas visíveis na base

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("dados/clientes.csv", sep=";", encoding="utf-8")

print(df.dtypes)
display(df.head(5))

id_item                    int64
id_pedido                  int64
id_cliente                 int64
idade_cliente              int64
cidade_cliente               str
estado_cliente               str
data_compra                  str
produto                      str
categoria_produto            str
quantidade                 int64
preco_unitario           float64
valor_total              float64
forma_pagamento              str
canal_compra                 str
dispositivo_utilizado        str
status_pedido                str
avaliacao_compra         float64
utilizou_cupom               str
valor_frete              float64
dtype: object


,id_item,id_pedido,id_cliente,idade_cliente,cidade_cliente,estado_cliente,data_compra,produto,categoria_produto,quantidade,preco_unitario,valor_total,forma_pagamento,canal_compra,dispositivo_utilizado,status_pedido,avaliacao_compra,utilizou_cupom,valor_frete
0,1,100001,118,56,Rio de Janeiro,RJ,2021-06-09 00:00:00,Alim_087_Standard,Alimentos,1,133.17,133.17,Boleto,Marketplace,Smartphone,Concluído,3.0,Não,10.47
1,2,100001,118,56,Rio de Janeiro,RJ,2021-06-09 00:00:00,Bele_003_Standard,Beleza,1,38.83,38.83,Boleto,Marketplace,Smartphone,Concluído,3.0,Não,10.47
2,3,100001,118,56,Rio de Janeiro,RJ,2021-06-09 00:00:00,Casa_052_Pro,Casa e Decoração,1,616.33,616.33,Boleto,Marketplace,Smartphone,Concluído,3.0,Não,10.47
3,4,100002,818,29,Aparecida de Goiânia,GO,2022-03-04 00:00:00,Moda_050_Pro,Moda,2,147.06,294.12,Cartão de Crédito,Marketplace,Desktop,Concluído,5.0,Não,19.16
4,5,100003,468,25,Feira de Santana,BA,2022-04-10 00:00:00,Bele_102_Standard,Beleza,1,147.45,147.45,Cartão de Crédito,App Mobile,Tablet,Concluído,6.0,Sim,0.39


In [13]:
# len() conta quantas linhas (registros) a base tem
print(f"Registros: {len(df)}")
# len() em df.columns conta quantas colunas a base tem
print(f"Colunas: {len(df.columns)}\n")
# df.info()

# Valores ausentes por coluna
print(df.isna().sum())

# Estatísticas das colunas numéricas (procure mínimos e máximos estranhos)
print(df.describe())

# Quantidade de linhas com quantidade zero ou negativa
print(f"\nQuantidade de linhas com vlaor negativo ou zero: {(df["quantidade"] <= 0).sum()}")

# Linhas duplicadas e id_item repetido
print("Linhas duplicadas:", df.duplicated().sum())
print("id_item repetido:", df["id_item"].duplicated().sum())

Registros: 50000
Colunas: 19

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_item                50000 non-null  int64  
 1   id_pedido              50000 non-null  int64  
 2   id_cliente             50000 non-null  int64  
 3   idade_cliente          50000 non-null  int64  
 4   cidade_cliente         49850 non-null  str    
 5   estado_cliente         49850 non-null  str    
 6   data_compra            50000 non-null  str    
 7   produto                50000 non-null  str    
 8   categoria_produto      50000 non-null  str    
 9   quantidade             50000 non-null  int64  
 10  preco_unitario         50000 non-null  float64
 11  valor_total            50000 non-null  float64
 12  forma_pagamento        50000 non-null  str    
 13  canal_compra           50000 non-null  str    
 14  dispositivo_utilizado  50000 non-nu

## 2. Tratamento de Inconsistências:
Identifique e trate valores ausentes, registros duplicados e quaisquer erros lógicos ou de digitação encontrados. Documente passo a passo as regras de limpeza aplicadas.

In [36]:
df_limpo = pd.read_csv("dados/clientes.csv", sep=";", encoding="utf-8")

print("Linhas antes da limpeza:", len(df_limpo))

Linhas antes da limpeza: 50000


In [37]:
# Remove linhas totalmente repetidas (mesmo registro aparecendo duas vezes)
df_limpo = df_limpo.drop_duplicates()
print("Linhas após remover duplicatas:", len(df_limpo))

Linhas após remover duplicatas: 49850


In [38]:
# Corrigindo colunas com nomes fixos e iguais ("Casa Decoracao" para "Casa e Decoração") * .replace faz isso
df_limpo["categoria_produto"] = df_limpo["categoria_produto"].replace(
    {"Casa Decoracao": "Casa e Decoração", "Eletronicos": "Eletrônicos"})

# Corrige a forma de pagamento escrita errado
df_limpo["forma_pagamento"] = df_limpo["forma_pagamento"].replace(
    {"Cartao Credito": "Cartão de Crédito"})

In [39]:
# Conta quantas linhas têm quantidade zero ou negativa (não existe comprar -10 unidades)
print("Quantidade inválida:", (df_limpo["quantidade"] <= 0).sum())

# Mantém só as linhas com quantidade maior que zero
# reset_index refaz a numeração das linhas (0, 1, 2...) depois de remover algumas
df_limpo = df_limpo[df_limpo["quantidade"] > 0].reset_index(drop=True)
print("Linhas após remover quantidade inválida:", len(df_limpo))

Quantidade inválida: 82
Linhas após remover quantidade inválida: 49768


In [40]:
# Converte texto para data; as datas inválidas viram NaT (data nula)
df_limpo["data_compra"] = pd.to_datetime(df_limpo["data_compra"], errors="coerce")
print("Datas inválidas:", df_limpo["data_compra"].isna().sum())

# Todos os itens de um pedido têm a mesma data, então copia a data de outro item do mesmo pedido
df_limpo["data_compra"] = df_limpo["data_compra"].fillna(
    df_limpo.groupby("id_pedido")["data_compra"].transform("first"))
print("Datas ainda nulas:", df_limpo["data_compra"].isna().sum())

Datas inválidas: 149
Datas ainda nulas: 43


In [41]:
# Contando quantas idades tem mais de 100 anos
print("Idades inválidas:", (df_limpo["idade_cliente"] > 100).sum())

# Mantém só idades até 100; o resto vira nulo
df_limpo["idade_cliente"] = df_limpo["idade_cliente"].where(df_limpo["idade_cliente"] <= 100)

# A idade é do cliente, então copiei de outra compra do mesmo cliente e colei na que estava vazia
df_limpo["idade_cliente"] = df_limpo["idade_cliente"].fillna(
    df_limpo.groupby("id_cliente")["idade_cliente"].transform("first"))

Idades inválidas: 68


In [42]:
# Cidade e estado também são do cliente: copia de outra compra dele
# O que não der para recuperar vira "Não informado"
for linha in ["cidade_cliente", "estado_cliente"]:
    df_limpo[linha] = df_limpo[linha].fillna(df_limpo.groupby("id_cliente")[linha].transform("first"))
    df_limpo[linha] = df_limpo[linha].fillna("Não informado")

In [43]:
print("Notas inválidas:", (df_limpo["avaliacao_compra"] > 5).sum() + (df_limpo["avaliacao_compra"] < 1).sum())

df_limpo["avaliacao_compra"] = df_limpo["avaliacao_compra"].where(df_limpo["avaliacao_compra"] <= 5)
df_limpo["avaliacao_compra"] = df_limpo["avaliacao_compra"].where(df_limpo["avaliacao_compra"] >= 1)

Notas inválidas: 150


In [44]:
print("Linhas depois da limpeza:", len(df_limpo))
print(df_limpo.isna().sum())
print("Quantidade mínima:", df_limpo["quantidade"].min())
print("Idade máxima:", df_limpo["idade_cliente"].max())

df_limpo.to_csv("dados/clientes_limpo.csv", sep=";", index=False, encoding="utf-8")

Linhas depois da limpeza: 49768
id_item                      0
id_pedido                    0
id_cliente                   0
idade_cliente                1
cidade_cliente               0
estado_cliente               0
data_compra                 43
produto                      0
categoria_produto            0
quantidade                   0
preco_unitario               0
valor_total                  0
forma_pagamento              0
canal_compra                 0
dispositivo_utilizado        0
status_pedido                0
avaliacao_compra         14409
utilizou_cupom               0
valor_frete                  0
dtype: int64
Quantidade mínima: 1
Idade máxima: 80.0
